# Анализ датасета `ai_student` (без визуализации)

Влияние использования генеративного ИИ на успеваемость студентов.

**План:**
1. Основные статистические показатели
2. Первичный обзор данных
3. Качество данных (пропуски, дубликаты)
4. Категориальные признаки
5. Корреляции
6. Групповой анализ

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('ai_student.csv')

## 0. Обзор таблицы: размерность и столбцы

In [2]:
# Размерность таблицы
print(f'Размерность: {df.shape[0]} строк (студентов), {df.shape[1]} столбцов\n')

# Описание всех столбцов
descriptions = {
    'Student_ID': 'Уникальный идентификатор студента',
    'Major_Category': 'Категория специальности (STEM, Business, Humanities, Medical, Arts)',
    'Year_of_Study': 'Курс обучения (Freshman/Sophomore/Junior/Senior)',
    'Pre_Semester_GPA': 'GPA до начала семестра (0–4)',
    'Weekly_GenAI_Hours': 'Часов в неделю работы с генеративным ИИ',
    'Primary_Use_Case': 'Основной сценарий использования ИИ',
    'Prompt_Engineering_Skill': 'Уровень навыка промпт-инжиниринга (Beginner/Intermediate/Advanced)',
    'Tool_Diversity': 'Разнообразие используемых ИИ-инструментов (1–5)',
    'Paid_Subscription': 'Наличие платной подписки (True/False)',
    'Traditional_Study_Hours': 'Часов в неделю традиционной учёбы (без ИИ)',
    'Perceived_AI_Dependency': 'Субъективная зависимость от ИИ (шкала)',
    'Institutional_Policy': 'Политика вуза по использованию ИИ',
    'Anxiety_Level_During_Exams': 'Уровень тревожности на экзаменах (шкала)',
    'Post_Semester_GPA': 'GPA после семестра (0–4)',
    'Skill_Retention_Score': 'Оценка удержания навыков (0–100)',
    'Burnout_Risk_Level': 'Уровень риска выгорания (Low/Medium/High)',
}

overview = pd.DataFrame({
    'Тип': df.dtypes.astype(str),
    'Непустых': df.notnull().sum(),
    'Уникальных': df.nunique(),
    'Описание': pd.Series(descriptions),
})
overview

Размерность: 50000 строк (студентов), 16 столбцов



,Тип,Непустых,Уникальных,Описание
Student_ID,int64,50000,50000,Уникальный идентификатор студента
Major_Category,object,50000,5,"Категория специальности (STEM, Business, Human..."
Year_of_Study,object,50000,5,Курс обучения (Freshman/Sophomore/Junior/Senior)
Pre_Semester_GPA,float64,50000,2389,GPA до начала семестра (0–4)
Weekly_GenAI_Hours,float64,50000,3566,Часов в неделю работы с генеративным ИИ
Primary_Use_Case,object,50000,5,Основной сценарий использования ИИ
Prompt_Engineering_Skill,object,50000,3,Уровень навыка промпт-инжиниринга (Beginner/In...
Tool_Diversity,int64,50000,5,Разнообразие используемых ИИ-инструментов (1–5)
Paid_Subscription,bool,50000,2,Наличие платной подписки (True/False)
Traditional_Study_Hours,float64,50000,2516,Часов в неделю традиционной учёбы (без ИИ)


## 1. Основные статистические показатели

In [3]:
# Описательная статистика числовых признаков:
# count — количество, mean — среднее, std — стандартное отклонение,
# min/max — минимум/максимум, 25%/50%/75% — квартили (50% = медиана)
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Student_ID,50000.0,125000.500000,14433.901067,100001.000,112500.75000,125000.500,137500.250,150000.000
Pre_Semester_GPA,50000.0,3.146102,0.478854,1.183,2.83400,3.210,3.521,3.998
Weekly_GenAI_Hours,50000.0,8.427752,8.269490,0.000,2.39000,5.800,11.720,40.000
Tool_Diversity,50000.0,2.800260,1.188020,1.000,2.00000,3.000,4.000,5.000
Traditional_Study_Hours,50000.0,11.209271,5.156426,1.000,7.56000,11.180,14.710,35.860
Perceived_AI_Dependency,50000.0,3.505360,1.820812,1.000,2.00000,3.000,5.000,10.000
Anxiety_Level_During_Exams,50000.0,4.270760,2.144066,1.000,3.00000,4.000,6.000,10.000
Post_Semester_GPA,50000.0,3.349299,0.495673,1.000,3.02375,3.421,3.749,4.000
Skill_Retention_Score,50000.0,75.798125,13.281626,10.780,66.82000,76.000,85.190,100.000


In [4]:
# Дополнительные показатели для числовых признаков
num = df.select_dtypes(include='number')
stats = pd.DataFrame({
    'mean': num.mean(),
    'median': num.median(),
    'std': num.std(),
    'min': num.min(),
    'max': num.max(),
    'range': num.max() - num.min(),
    'skew': num.skew(),       # асимметрия
    'kurtosis': num.kurt(),   # эксцесс
})
stats.round(3)

,mean,median,std,min,max,range,skew,kurtosis
Student_ID,125000.500,125000.500,14433.901,100001.000,150000.000,49999.000,0.000,-1.200
Pre_Semester_GPA,3.146,3.210,0.479,1.183,3.998,2.815,-0.602,-0.105
Weekly_GenAI_Hours,8.428,5.800,8.269,0.000,40.000,40.000,1.610,2.557
Tool_Diversity,2.800,3.000,1.188,1.000,5.000,4.000,0.167,-0.744
Traditional_Study_Hours,11.209,11.180,5.156,1.000,35.860,34.860,0.130,-0.305
Perceived_AI_Dependency,3.505,3.000,1.821,1.000,10.000,9.000,0.655,0.263
Anxiety_Level_During_Exams,4.271,4.000,2.144,1.000,10.000,9.000,0.362,-0.408
Post_Semester_GPA,3.349,3.421,0.496,1.000,4.000,3.000,-0.675,-0.034
Skill_Retention_Score,75.798,76.000,13.282,10.780,100.000,89.220,-0.215,-0.217


## 2. Первичный обзор

In [5]:
print('Размерность (строки, столбцы):', df.shape)
df.head()

Размерность (строки, столбцы): (50000, 16)


,Student_ID,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,100001,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,True,8.13,5,Allowed_With_Citation,6,2.393,86.44,High
1,100002,Medical,Junior,3.821,1.12,Ideation,Advanced,5,False,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low
2,100003,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,False,10.35,5,Strict_Ban,9,3.499,73.93,Medium
3,100004,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,False,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium
4,100005,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,False,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student_ID                  50000 non-null  int64  
 1   Major_Category              50000 non-null  object 
 2   Year_of_Study               50000 non-null  object 
 3   Pre_Semester_GPA            50000 non-null  float64
 4   Weekly_GenAI_Hours          50000 non-null  float64
 5   Primary_Use_Case            50000 non-null  object 
 6   Prompt_Engineering_Skill    50000 non-null  object 
 7   Tool_Diversity              50000 non-null  int64  
 8   Paid_Subscription           50000 non-null  bool   
 9   Traditional_Study_Hours     50000 non-null  float64
 10  Perceived_AI_Dependency     50000 non-null  int64  
 11  Institutional_Policy        50000 non-null  object 
 12  Anxiety_Level_During_Exams  50000 non-null  int64  
 13  Post_Semester_GPA           500

## 3. Качество данных: пропуски и дубликаты

In [7]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
quality = pd.DataFrame({'Пропусков': missing, 'Доля, %': missing_pct})
print('Пропущенные значения по столбцам:')
print(quality[quality['Пропусков'] > 0] if quality['Пропусков'].sum() else 'Пропусков нет')
print('\nПолных дубликатов строк:', df.duplicated().sum())
print('Дубликатов по Student_ID:', df['Student_ID'].duplicated().sum())

Пропущенные значения по столбцам:
Пропусков нет

Полных дубликатов строк: 0
Дубликатов по Student_ID: 0


## 4. Категориальные признаки

In [8]:
# Сводка: количество уникальных значений, самое частое и его частота
df.describe(include='object').T

,count,unique,top,freq
Major_Category,50000,5,STEM,15059
Year_of_Study,50000,5,Junior,11045
Primary_Use_Case,50000,5,Debugging/Troubleshooting,12295
Prompt_Engineering_Skill,50000,3,Beginner,18495
Institutional_Policy,50000,3,Allowed_With_Citation,25224
Burnout_Risk_Level,50000,3,Medium,21144


In [9]:
cat_cols = ['Major_Category', 'Year_of_Study', 'Primary_Use_Case',
            'Prompt_Engineering_Skill', 'Paid_Subscription',
            'Institutional_Policy', 'Burnout_Risk_Level']

for col in cat_cols:
    print(f'\n=== {col} ===')
    print(df[col].value_counts(dropna=False))


=== Major_Category ===
STEM          15059
Business      12538
Humanities     9994
Medical        6476
Arts           5933
Name: Major_Category, dtype: int64

=== Year_of_Study ===
Junior       11045
Freshman     11031
Senior       10634
Sophomore     9860
Graduate      7430
Name: Year_of_Study, dtype: int64

=== Primary_Use_Case ===
Debugging/Troubleshooting    12295
Copywriting/Drafting         12011
Ideation                     10721
Summarizing_Reading           8633
Direct_Answer_Generation      6340
Name: Primary_Use_Case, dtype: int64

=== Prompt_Engineering_Skill ===
Beginner        18495
Intermediate    17696
Advanced        13809
Name: Prompt_Engineering_Skill, dtype: int64

=== Paid_Subscription ===
False    28846
True     21154
Name: Paid_Subscription, dtype: int64

=== Institutional_Policy ===
Allowed_With_Citation    25224
Actively_Encouraged      14988
Strict_Ban                9788
Name: Institutional_Policy, dtype: int64

=== Burnout_Risk_Level ===
Medium    21144
Low

## 5. Корреляции

Создаём признак `GPA_Change = Post_Semester_GPA - Pre_Semester_GPA` и смотрим, с чем он связан.

In [10]:
df['GPA_Change'] = df['Post_Semester_GPA'] - df['Pre_Semester_GPA']

# Корреляционная матрица числовых признаков
corr = df.select_dtypes(include='number').corr()
corr.round(2)

,Student_ID,Pre_Semester_GPA,Weekly_GenAI_Hours,Tool_Diversity,Traditional_Study_Hours,Perceived_AI_Dependency,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,GPA_Change
Student_ID,1.00,0.00,0.00,-0.00,0.00,0.00,0.01,-0.00,-0.00,-0.01
Pre_Semester_GPA,0.00,1.00,-0.00,-0.01,-0.00,0.00,-0.00,0.93,0.10,-0.10
Weekly_GenAI_Hours,0.00,-0.00,1.00,0.01,-0.16,0.67,0.27,-0.02,-0.12,-0.05
Tool_Diversity,-0.00,-0.01,0.01,1.00,0.00,0.01,0.00,0.03,0.20,0.08
Traditional_Study_Hours,0.00,-0.00,-0.16,0.00,1.00,-0.10,-0.04,0.14,0.15,0.38
Perceived_AI_Dependency,0.00,0.00,0.67,0.01,-0.10,1.00,0.31,-0.01,-0.08,-0.04
Anxiety_Level_During_Exams,0.01,-0.00,0.27,0.00,-0.04,0.31,1.00,-0.02,-0.04,-0.04
Post_Semester_GPA,-0.00,0.93,-0.02,0.03,0.14,-0.01,-0.02,1.00,0.17,0.28
Skill_Retention_Score,-0.00,0.10,-0.12,0.20,0.15,-0.08,-0.04,0.17,1.00,0.20
GPA_Change,-0.01,-0.10,-0.05,0.08,0.38,-0.04,-0.04,0.28,0.20,1.00


In [11]:
# С чем сильнее всего связано изменение GPA (GPA_Change = Post - Pre, т.е. ПРИРОСТ балла)
target_corr = corr['GPA_Change'].drop('GPA_Change').sort_values(key=abs, ascending=False)

print('Корреляция признаков с приростом GPA (GPA_Change):')
print('  знак + : чем больше признак, тем СИЛЬНЕЕ повышение GPA')
print('  знак - : чем больше признак, тем СЛАБЕЕ повышение (прирост меньше)\n')

for feat, r in target_corr.items():
    direction = '↑ сильнее повышение' if r > 0 else '↓ слабее повышение'
    print(f'  {feat:<28} {r:+.3f}  {direction}')

Корреляция признаков с приростом GPA (GPA_Change):
  знак + : чем больше признак, тем СИЛЬНЕЕ повышение GPA
  знак - : чем больше признак, тем СЛАБЕЕ повышение (прирост меньше)

  Traditional_Study_Hours      +0.376  ↑ сильнее повышение
  Post_Semester_GPA            +0.277  ↑ сильнее повышение
  Skill_Retention_Score        +0.196  ↑ сильнее повышение
  Pre_Semester_GPA             -0.104  ↓ слабее повышение
  Tool_Diversity               +0.081  ↑ сильнее повышение
  Weekly_GenAI_Hours           -0.046  ↓ слабее повышение
  Anxiety_Level_During_Exams   -0.040  ↓ слабее повышение
  Perceived_AI_Dependency      -0.039  ↓ слабее повышение
  Student_ID                   -0.007  ↓ слабее повышение


## 6. Групповой анализ

In [12]:
# Средние показатели по уровню навыка промпт-инжиниринга
skill_order = ['Beginner', 'Intermediate', 'Advanced']
agg = (df.groupby('Prompt_Engineering_Skill')
         [['Weekly_GenAI_Hours', 'GPA_Change', 'Skill_Retention_Score', 'Anxiety_Level_During_Exams']]
         .mean()
         .reindex(skill_order))
print('Средние показатели по уровню навыка промпт-инжиниринга:')
agg.round(3)

Средние показатели по уровню навыка промпт-инжиниринга:


,Weekly_GenAI_Hours,GPA_Change,Skill_Retention_Score,Anxiety_Level_During_Exams
Prompt_Engineering_Skill,,,,
Beginner,8.286,0.185,71.101,4.263
Intermediate,8.367,0.187,75.825,4.273
Advanced,8.694,0.248,82.054,4.278


In [13]:
# Среднее изменение GPA по политике вуза и по специальности
print('Среднее изменение GPA по политике вуза:')
print(df.groupby('Institutional_Policy')['GPA_Change'].mean().round(3).sort_values(ascending=False))
print('\nСреднее изменение GPA по специальности:')
print(df.groupby('Major_Category')['GPA_Change'].mean().round(3).sort_values(ascending=False))

Среднее изменение GPA по политике вуза:
Institutional_Policy
Allowed_With_Citation    0.208
Actively_Encouraged      0.206
Strict_Ban               0.187
Name: GPA_Change, dtype: float64

Среднее изменение GPA по специальности:
Major_Category
STEM          0.217
Medical       0.201
Humanities    0.198
Arts          0.197
Business      0.194
Name: GPA_Change, dtype: float64


In [14]:
# Средние показатели по уровню риска выгорания
burnout_order = ['Low', 'Medium', 'High']
burn = (df.groupby('Burnout_Risk_Level')
          [['Weekly_GenAI_Hours', 'Perceived_AI_Dependency',
            'Anxiety_Level_During_Exams', 'Traditional_Study_Hours', 'GPA_Change']]
          .mean()
          .reindex(burnout_order))
print('Средние показатели по уровню риска выгорания:')
burn.round(3)

Средние показатели по уровню риска выгорания:


,Weekly_GenAI_Hours,Perceived_AI_Dependency,Anxiety_Level_During_Exams,Traditional_Study_Hours,GPA_Change
Burnout_Risk_Level,,,,,
Low,4.644,2.820,3.928,11.966,0.200
Medium,7.349,3.365,4.170,11.290,0.210
High,15.215,4.642,4.889,10.082,0.196


## 7. Связь платной подписки и GPA

Проверяем гипотезу: влияет ли наличие платной подписки (`Paid_Subscription`) на успеваемость.
Сравниваем группы по среднему GPA и приросту GPA, а значимость различий оцениваем t-критерием.

In [15]:
from scipy import stats

# Размеры групп
print('Размер групп (Paid_Subscription):')
print(df['Paid_Subscription'].value_counts(), '\n')

# Средние значения GPA по группам
grp = (df.groupby('Paid_Subscription')
         [['Pre_Semester_GPA', 'Post_Semester_GPA', 'GPA_Change']]
         .agg(['mean', 'std', 'count']))
print('Статистика GPA по наличию платной подписки:')
display(grp.round(4))

paid = df[df['Paid_Subscription'] == True]
free = df[df['Paid_Subscription'] == False]

# t-тест Уэлча (не требует равных дисперсий) + размер эффекта (корреляция)
print('\nПроверка значимости различий (t-тест Уэлча):')
for col in ['Post_Semester_GPA', 'GPA_Change']:
    t, p = stats.ttest_ind(paid[col], free[col], equal_var=False)
    diff = paid[col].mean() - free[col].mean()
    corr = df[col].corr(df['Paid_Subscription'].astype(int))
    sig = 'значимо' if p < 0.05 else 'НЕ значимо'
    print(f'  {col}: разница (paid - free) = {diff:+.4f}, '
          f't = {t:.3f}, p = {p:.4g} -> {sig}; корреляция = {corr:+.4f}')

Размер групп (Paid_Subscription):
False    28846
True     21154
Name: Paid_Subscription, dtype: int64 

Статистика GPA по наличию платной подписки:


Pre_Semester_GPA                Post_Semester_GPA                GPA_Change               
                              mean     std  count              mean     std  count       mean     std  count
Paid_Subscription                                                                                           
False                       3.1465  0.4791  28846            3.3470  0.4942  28846     0.2004  0.1818  28846
True                        3.1455  0.4785  21154            3.3525  0.4977  21154     0.2070  0.1942  21154


Проверка значимости различий (t-тест Уэлча):
  Post_Semester_GPA: разница (paid - free) = +0.0055, t = 1.235, p = 0.2169 -> НЕ значимо; корреляция = +0.0055
  GPA_Change: разница (paid - free) = +0.0065, t = 3.822, p = 0.0001324 -> значимо; корреляция = +0.0173


## 8. Связь разнообразия инструментов (`Tool_Diversity`) и GPA

`Tool_Diversity` — порядковый признак (1–5: сколько разных ИИ-инструментов использует студент).
Проверяем корреляцию с GPA: Пирсона (линейная) и Спирмена (монотонная, устойчивее для порядковых шкал).

In [16]:
# Корреляция Tool_Diversity с GPA
print('Корреляция Tool_Diversity с GPA:')
for col in ['Pre_Semester_GPA', 'Post_Semester_GPA', 'GPA_Change']:
    r, p = stats.pearsonr(df['Tool_Diversity'], df[col])
    rho, ps = stats.spearmanr(df['Tool_Diversity'], df[col])
    print(f'  {col}: Pearson r = {r:+.4f} (p={p:.3g}) | Spearman rho = {rho:+.4f} (p={ps:.3g})')

# Средний GPA по уровню разнообразия инструментов
print('\nСредний GPA по уровню Tool_Diversity:')
display(df.groupby('Tool_Diversity')[['Post_Semester_GPA', 'GPA_Change']].mean().round(4))

Корреляция Tool_Diversity с GPA:
  Pre_Semester_GPA: Pearson r = -0.0057 (p=0.205) | Spearman rho = -0.0039 (p=0.389)
  Post_Semester_GPA: Pearson r = +0.0253 (p=1.6e-08) | Spearman rho = +0.0296 (p=3.81e-11)
  GPA_Change: Pearson r = +0.0814 (p=2.82e-74) | Spearman rho = +0.0803 (p=2.45e-72)

Средний GPA по уровню Tool_Diversity:


,Post_Semester_GPA,GPA_Change
Tool_Diversity,,
1,3.3330,0.1821
2,3.3335,0.1886
3,3.3549,0.2071
4,3.3686,0.2243
5,3.3628,0.2249


## 9. `Weekly_GenAI_Hours` — интенсивность использования ИИ

Количество часов в неделю, которое студент тратит на работу с генеративным ИИ.
Смотрим распределение, корреляции с ключевыми метриками и поведение по квинтилям
(чтобы поймать возможную нелинейность — эффект «слишком много ИИ»).

In [17]:
w = df['Weekly_GenAI_Hours']

# Распределение
print('Распределение Weekly_GenAI_Hours:')
print(w.describe().round(2))
print('Асимметрия (skew):', round(w.skew(), 3), '(правый хвост — есть студенты с очень большим числом часов)\n')

# Корреляции с ключевыми метриками
print('Корреляции Weekly_GenAI_Hours с другими признаками:')
for col in ['Post_Semester_GPA', 'GPA_Change', 'Skill_Retention_Score',
            'Anxiety_Level_During_Exams', 'Perceived_AI_Dependency', 'Traditional_Study_Hours']:
    r, p = stats.pearsonr(w, df[col])
    print(f'  {col}: r = {r:+.4f} (p={p:.3g})')

# Поведение по квинтилям часов (поиск нелинейности)
df['GenAI_Hours_Quintile'] = pd.qcut(w, 5)
print('\nСредние метрики по квинтилям часов GenAI:')
display(df.groupby('GenAI_Hours_Quintile')
          [['GPA_Change', 'Skill_Retention_Score', 'Anxiety_Level_During_Exams']]
          .mean().round(3))

# Средние часы по уровню выгорания
print('Средние часы GenAI по уровню риска выгорания:')
print(df.groupby('Burnout_Risk_Level')['Weekly_GenAI_Hours']
        .mean().round(2).reindex(['Low', 'Medium', 'High']))

Распределение Weekly_GenAI_Hours:
count    50000.00
mean         8.43
std          8.27
min          0.00
25%          2.39
50%          5.80
75%         11.72
max         40.00
Name: Weekly_GenAI_Hours, dtype: float64
Асимметрия (skew): 1.61 (правый хвост — есть студенты с очень большим числом часов)

Корреляции Weekly_GenAI_Hours с другими признаками:
  Post_Semester_GPA: r = -0.0186 (p=3.19e-05)
  GPA_Change: r = -0.0465 (p=2.52e-25)
  Skill_Retention_Score: r = -0.1181 (p=9.73e-155)
  Anxiety_Level_During_Exams: r = +0.2691 (p=0)
  Perceived_AI_Dependency: r = +0.6655 (p=0)
  Traditional_Study_Hours: r = -0.1574 (p=1.27e-274)

Средние метрики по квинтилям часов GenAI:


,GPA_Change,Skill_Retention_Score,Anxiety_Level_During_Exams
GenAI_Hours_Quintile,,,
"(-0.001, 1.85]",0.188,75.611,3.804
"(1.85, 4.25]",0.197,76.148,3.873
"(4.25, 7.68]",0.219,76.880,4.007
"(7.68, 13.59]",0.231,77.085,4.404
"(13.59, 40.0]",0.181,73.267,5.266


Средние часы GenAI по уровню риска выгорания:
Burnout_Risk_Level
Low        4.64
Medium     7.35
High      15.21
Name: Weekly_GenAI_Hours, dtype: float64


## 10. Связь специальности и навыка промпт-инжиниринга

`Major_Category` и `Prompt_Engineering_Skill` — оба категориальные.
Проверяем независимость критерием хи-квадрат (χ²); силу связи оцениваем через V Крамера.

In [18]:
skill_order = ['Beginner', 'Intermediate', 'Advanced']

# Таблица сопряжённости
ct = pd.crosstab(df['Major_Category'], df['Prompt_Engineering_Skill'])[skill_order]
print('Таблица сопряжённости (количество студентов):')
display(ct)

# Доли уровней навыка внутри каждой специальности
prop = pd.crosstab(df['Major_Category'], df['Prompt_Engineering_Skill'],
                   normalize='index')[skill_order] * 100
print('Распределение навыка внутри специальности, %:')
display(prop.round(1))

# Хи-квадрат + V Крамера (сила связи)
chi2, p, dof, _ = stats.chi2_contingency(ct)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
sig = 'есть связь' if p < 0.05 else 'независимы'
print(f'chi2 = {chi2:.1f}, dof = {dof}, p = {p:.3g} -> {sig}')
print(f"V Крамера = {cramers_v:.4f} (0 — нет связи, 1 — полная; <0.1 — очень слабая)")

Таблица сопряжённости (количество студентов):


Prompt_Engineering_Skill,Beginner,Intermediate,Advanced
Major_Category,,,
Arts,2277,2172,1484
Business,5090,4347,3101
Humanities,4007,3526,2461
Medical,2558,2228,1690
STEM,4563,5423,5073


Распределение навыка внутри специальности, %:


Prompt_Engineering_Skill,Beginner,Intermediate,Advanced
Major_Category,,,
Arts,38.4,36.6,25.0
Business,40.6,34.7,24.7
Humanities,40.1,35.3,24.6
Medical,39.5,34.4,26.1
STEM,30.3,36.0,33.7


chi2 = 565.3, dof = 8, p = 6.8e-117 -> есть связь
V Крамера = 0.0752 (0 — нет связи, 1 — полная; <0.1 — очень слабая)


## 11. Кто набирает Skill_Retention_Score = 100 и есть ли «не-пользователи» ИИ

Отдельного флага «пользуется ИИ» в данных нет, поэтому считаем не-пользователем студента с `Weekly_GenAI_Hours == 0`.
Смотрим, у кого максимальный балл удержания навыков и сколько вообще студентов не пользовались ИИ.

In [19]:
# Признак: пользовался ли студент ИИ
df['used_ai'] = df['Weekly_GenAI_Hours'] > 0

# Есть ли вообще не-пользователи ИИ
n_nonusers = (~df['used_ai']).sum()
print(f'Студентов с Weekly_GenAI_Hours == 0 (не пользовались ИИ): {n_nonusers} из {len(df)}')
print(f'Доля не-пользователей: {n_nonusers / len(df):.2%}\n')

# Кто набирает максимальный балл удержания навыков
top = df[df['Skill_Retention_Score'] == 100]
print(f'Студентов со Skill_Retention_Score == 100: {len(top)}')
print(f'  из них пользовались ИИ:    {top["used_ai"].sum()}')
print(f'  из них НЕ пользовались ИИ: {(~top["used_ai"]).sum()}')
print('\nЧасы GenAI у тех, у кого score == 100:')
print(top['Weekly_GenAI_Hours'].describe().round(2))

# Сравнение удержания навыков: пользователи vs не-пользователи
print('\nSkill_Retention_Score: пользователи vs не-пользователи ИИ:')
display(df.groupby('used_ai')['Skill_Retention_Score']
          .agg(['count', 'mean', 'min', 'max']).round(2))

Студентов с Weekly_GenAI_Hours == 0 (не пользовались ИИ): 28 из 50000
Доля не-пользователей: 0.06%

Студентов со Skill_Retention_Score == 100: 2146
  из них пользовались ИИ:    2145
  из них НЕ пользовались ИИ: 1

Часы GenAI у тех, у кого score == 100:
count    2146.00
mean        7.95
std         6.73
min         0.00
25%         2.88
50%         6.34
75%        11.37
max        40.00
Name: Weekly_GenAI_Hours, dtype: float64

Skill_Retention_Score: пользователи vs не-пользователи ИИ:


,count,mean,min,max
used_ai,,,,
False,28,77.61,59.00,100.0
True,49972,75.80,10.78,100.0


## 12. Итоги анализа

**Данные:** 50 000 студентов, 16 признаков, пропусков и дубликатов нет — датасет чистый.

**Ключевые выводы:**

1. **GPA в среднем растёт.** Средний прирост `GPA_Change` = +0.20, у 87.5% студентов балл повысился за семестр.

2. **Главный драйвер роста GPA — традиционная учёба** (корреляция +0.38), а не ИИ. Чем больше часов обычной учёбы, тем сильнее повышение GPA.

3. **Платная подписка на GPA не влияет.** Разница с бесплатными студентами +0.006 балла (p=0.22 для итогового GPA) — практически нулевая.

4. **Разнообразие инструментов (Tool_Diversity)** даёт слабый положительный эффект на прирост GPA (r=+0.08): от +0.18 при 1 инструменте до +0.22 при 4–5, эффект насыщается.

5. **Интенсивность ИИ (Weekly_GenAI_Hours) — нелинейная связь (перевёрнутая U):** оптимум ~8–13 ч/нед (макс. прирост GPA и удержание навыков). Свыше ~14 ч/нед — прирост падает, удержание навыков снижается (77→73), тревожность растёт (4.4→5.3).

6. **Чрезмерный ИИ → выгорание.** У группы High риска выгорания в среднем 15 ч/нед против 4.6 ч/нед у Low. Также сильная связь часов ИИ с зависимостью от него (r=+0.67) и тревожностью (r=+0.27).

7. **Специальность и навык промптинга связаны слабо** (V Крамера=0.075) — связь держится почти целиком на STEM (больше Advanced: 34% против ~25% у остальных). Между не-STEM направлениями разницы нет.

8. **Почти все пользуются ИИ:** не-пользователей всего 28 из 50 000. Поэтому максимальный балл удержания навыков (=100) набирают «пользователи» лишь в силу их подавляющего большинства, а не из-за пользы ИИ.

**Главный вывод:** ИИ полезен в умеренных дозах (≈1–2 ч/день) и в связке с традиционной учёбой; чрезмерное использование (>2 ч/день) вредит и успеваемости, и навыкам, и психологическому состоянию.